# Middleware in LangChain

Middleware in LangChain allows you to intercept and modify the flow of requests and responses between your application and the language model. It acts as a layer that runs **before** and **after** the model execution, enabling you to add custom logic without changing your core application code. Middleware is useful for tasks such as logging, authentication, caching, rate limiting, monitoring, and modifying prompts or model responses.

By using middleware, developers can keep their applications modular and maintainable. Multiple middleware components can be chained together, with each one performing a specific responsibility before passing the request to the next layer. This makes it easier to implement cross-cutting concerns that should apply to every model invocation.

#### When to Use

- Logging requests and responses
- Authentication and authorization
- Prompt modification
- Response filtering
- Token usage tracking
- Rate limiting
- Caching model responses
- Monitoring and analytics
- Error handling
- Custom validation

#### Example

```python
from langchain.agents.middleware import wrap_model_call

@wrap_model_call
def log_request(request, handler):
    print("User Prompt:", request.messages[-1].content)

    response = handler(request)

    print("AI Response:", response.content)
    return response
```

In this example, the middleware logs the user's prompt before the model runs and prints the AI's response after the model finishes generating its answer.

In [59]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.0,
    max_retries=2,
    # other params...
)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.13'}}, output_version=None, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002C1D5676EA0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002C1D5677820>, model_name='openai/gpt-oss-120b', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

# Summarization Middleware in LangChain

A **Summarization Middleware** is used to automatically summarize long conversation histories before they are sent to the language model. As a conversation grows, sending every previous message increases token usage and cost. Instead of passing the entire history, the middleware replaces older messages with a concise summary while preserving the important context. This helps the model stay within its context window and improves performance.

Summarization middleware is especially useful for long-running chatbots, AI assistants, customer support systems, and agent workflows where conversations can span hundreds of messages. By keeping only recent messages and a summary of earlier interactions, applications can maintain context while reducing latency, token consumption, and API costs.

#### When to Use

- Long conversations
- AI chatbots
- Customer support assistants
- AI agents with memory
- Reducing token usage and cost
- Staying within the model's context window
- Improving performance in long-running sessions

#### Example

```python
from langchain.agents.middleware import SummarizationMiddleware

middleware = SummarizationMiddleware(
    model=model,
    max_tokens_before_summary=4000,
    keep_last_n_messages=10,
)
```

**How it works:**

1. The conversation starts normally.
2. As more messages are added, the token count increases.
3. When the token limit is reached, the middleware summarizes the older messages.
4. The summary replaces the old conversation, while the most recent messages are kept.
5. The model receives the summary along with the latest messages, allowing it to continue the conversation with the necessary context while using fewer tokens.

In [60]:

from langchain_core.messages import SystemMessage,HumanMessage,AIMessage,ToolMessage

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=model,
    tools=[],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages", 10),
            keep=("messages", 5),
        )
    ],
)

In [61]:
### Run with Thread

config ={"configurable":{"thread_id": "thread_1"}}

In [62]:
questions = [
    "What is the capital of France?",
    "What is the population of Paris?",
    "What is the largest city in France?",
    "What is the climate like in Paris?",
    "What is the history of France? in 200 characters",
    "What is 5 multiplied by 6?",
    "What is 10 divided by 2?",
]

for q in questions:
    res = agent.invoke(
        {"messages": [HumanMessage(content=q)]},
        config=config,
    )

    print(f"Question: {q}")
    print(f"Answer: {res['messages'][-1].content}")
    print(f"Total Messages: {len(res['messages'])}")
    print("-" * 50)

Question: What is the capital of France?
Answer: The capital of France is **Paris**.
Total Messages: 2
--------------------------------------------------
Question: What is the population of Paris?
Answer: As of the most recent official estimates (2023‑2024), the **city of Paris proper** has a population of about **2.1 million people**.  

- **Paris intra‑muros (city limits)**: ~2,110,000 residents.  
- **Paris metropolitan area (Île‑de‑France region)**: roughly **12 million** people, making it one of the largest urban agglomerations in Europe.  

These figures come from France’s national statistics office (INSEE) and are updated annually. Keep in mind that the population can fluctuate slightly due to migration, births, and deaths, so newer releases may show minor changes.
Total Messages: 4
--------------------------------------------------
Question: What is the largest city in France?
Answer: The largest city in France by population is **Paris**.  

- **Population (city proper)**: abou

### Bases On token size

In [63]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver


@tool
def search_hotel(city: str) -> str:
    """search Hotels - returns long response to use more tokens"""
    return f"""
Here are some recommended hotels in {city}:

1. Grand Palace Hotel ⭐⭐⭐⭐⭐
   - Luxury hotel with premium rooms, spa, rooftop pool, and fine dining.
   - Starting from ₹8,500 per night.

2. City View Residency ⭐⭐⭐⭐
   - Comfortable rooms, complimentary breakfast, free Wi-Fi, and central location.
   - Starting from ₹4,200 per night.

3. Green Leaf Inn ⭐⭐⭐⭐
   - Peaceful stay with a garden, family-friendly rooms, and excellent service.
   - Starting from ₹3,800 per night.

4. Royal Comfort Suites ⭐⭐⭐⭐
   - Modern suites with a fitness center, business lounge, and airport shuttle.
   - Starting from ₹5,500 per night.

5. Budget Stay Express ⭐⭐⭐
   - Affordable accommodation with clean rooms, free parking, and 24/7 reception.
   - Starting from ₹2,000 per night.
"""
    pass


agent = create_agent(
    model=model,
    tools=[search_hotel],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens", 2500),
            keep=("tokens", 200),
        )
    ],
)

config ={"configurable":{"thread_id": "thread_1"}}
def count_tokens(message):
    total =  sum(len(str(m.content)) for m in message)
    return total/4

In [64]:
cities = ["paris", "london", "tokyo","new Delhi"]
for city in cities:
    res = agent.invoke({"messages":[HumanMessage(content=f"Search for hotels in {city}")]}, config=config)
    tokens = count_tokens(res["messages"])
    print(f"{city}: {tokens} tokens, {len(res['messages'])} messages")
    print(f"{res['messages']}")

paris: 667.0 tokens, 4 messages
[HumanMessage(content='Search for hotels in paris', additional_kwargs={}, response_metadata={}, id='89f3588c-ebcf-49b2-98f5-f2c8a381cb88'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to search for hotels in Paris. Use the provided function search_hotel with city "Paris".', 'tool_calls': [{'id': 'fc_b7966d75-eb16-440e-be03-b0cc4712c950', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotel'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 130, 'total_tokens': 181, 'completion_time': 0.105854718, 'completion_tokens_details': {'reasoning_tokens': 23}, 'prompt_time': 0.004919567, 'prompt_tokens_details': None, 'queue_time': 0.28608942, 'total_time': 0.110774285}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_5082008e34', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01kq831qwweq6a06hjrtfrm35e` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 6639, Requested 1473. Please try again in 840ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

# Human-in-the-Loop (HITL) in LangChain

**Human-in-the-Loop (HITL)** is a technique that allows a human to review, approve, reject, or modify an AI's actions before they are executed. Instead of allowing the AI to make important decisions on its own, the workflow pauses at specific points and waits for human input. This approach improves accuracy, safety, and accountability, especially for tasks that involve sensitive data or irreversible actions.

Human-in-the-Loop is commonly used in AI agents that interact with external systems such as email, databases, payment services, or production environments. Before sending an email, deleting a file, processing a payment, or executing code, the AI can ask for user confirmation. Once the user approves the action, the workflow continues; otherwise, it can be modified or cancelled.

## When to Use

- Sending emails
- Processing payments
- Deleting or modifying data
- Executing shell commands
- Deploying applications
- Approving AI-generated content
- Database updates
- Customer support workflows
- Any high-risk or irreversible action

## Workflow

```text
User Request
      │
      ▼
AI Agent
      │
      ▼
Plans an Action
      │
      ▼
⏸ Pause for Human Approval
      │
 ┌────┴────┐
 │         │
Approve   Reject/Edit
 │         │
 ▼         ▼
Execute   Cancel or Modify
```

## Example

```python
user: "Send an email to the client."

AI:
"I've drafted the email. Would you like me to send it?"

User:
"Yes."

AI:
✅ Email sent successfully.
```

Human-in-the-Loop helps build safer and more reliable AI applications by ensuring that important decisions are verified by a person before execution.

In [ ]:
from langchain_core.messages import SystemMessage,HumanMessage,AIMessage,ToolMessage
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware,HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
@tool
def read_email_tool(email_id: str) -> str:
    """Read an email by its ID."""
    # Placeholder implementation - replace with actual email reading logic
    return f"Content of email {email_id}"
@tool
def send_email_tool(email_id: str, content: str,subject: str,cc: list) -> str:
    """Send an email by its ID."""
    # Placeholder implementation - replace with actual email sending logic
    return f"Email {email_id} sent with subject: {subject}"

In [65]:
agent = create_agent(
    model=model,
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "reject", "edit", "respond"]
                },
                "read_email_tool": False
            }
        )
    ],
)

In [ ]:
config = {
    "configurable": {
        "thread_id": "thread_1"
    }
}

result = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Send an email to john@example.com with the body: Hello, how are you?"
            )
        ]
    },
    config=config
)

In [ ]:
result

{'messages': [HumanMessage(content='Send an email to john@example.com with the body: Hello, how are you?', additional_kwargs={}, response_metadata={}, id='152b6c00-75fc-40c1-8871-776596bb8dfe'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to send an email to john@example.com with body "Hello, how are you?". We need to use send_email_tool. It requires email_id, cc, content, subject. We don\'t have subject. Probably we need to set a default subject? The user didn\'t specify subject. Could set subject empty or "No Subject". Also need email_id: maybe we need to create a new email? The tool expects email_id of an existing email? The description: "Send an email by its ID." It likely means we have a draft email with that ID. But we don\'t have one. Could we first read? Not needed. Possibly we need to create a new email? There\'s no create tool. Might need to assume email_id is something like "new". Or we could ask clarification. But likely we can send with 

In [ ]:
if "__interrupt__" in result:
    print("Paused. Approving email...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "approve"
                    }
                ]
            }
        ),
        config=config,
    )

    print(result["messages"][-1].content)

In [ ]:
result

{'messages': [HumanMessage(content='Send an email to john@example.com with the body: Hello, how are you?', additional_kwargs={}, response_metadata={}, id='73a444b1-2e49-4c4c-8ace-fa4031969786'),
  AIMessage(content='I’m ready to send the email, but I need a couple of details first:\n\n1. **Subject line** – what would you like the email’s subject to be?\n2. **Email ID** – the system requires an existing email ID (e.g., a draft ID) to send from. Could you provide that ID, or let me know if you’d like me to create a new draft for you?', additional_kwargs={'reasoning_content': 'The user wants to send an email to john@example.com with body "Hello, how are you?". We need to use send_email_tool. It requires email_id, cc, content, subject. We don\'t have subject. Probably we need to set a default subject? The user didn\'t specify subject. Could set subject empty or "No Subject". Also need email_id: maybe we need to create a new email? The tool expects email_id of an existing email? The descrip

### Reject

In [66]:
config2 ={"configurable": {"thread_id": "test_reject_1"}}
result = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Send an email to john@example.com with the body: Hello, how are you?"
            )
        ]
    },
    config=config2
)

In [67]:
result

{'messages': [HumanMessage(content='Send an email to john@example.com with the body: Hello, how are you?', additional_kwargs={}, response_metadata={}, id='501c60a8-9ebf-4d57-aeb7-efc8987928cd'),
  AIMessage(content='I’m ready to send the email, but I need a couple of details first:\n\n1. **Subject line** – what would you like the subject of the email to be?\n2. **Email ID** – the sending tool requires an existing email ID (e.g., a draft you’ve created). If you have a specific email ID you’d like to use, please provide it; otherwise, let me know if you’d like me to create a new draft for you.', additional_kwargs={'reasoning_content': 'The user wants to send an email to john@example.com with body "Hello, how are you?". We need to use send_email_tool. It requires email_id, cc, content, subject. We don\'t have subject. Probably we need to set a default subject? The user didn\'t specify subject. Could set subject empty or "No Subject". Also need email_id: maybe we need to create a new email

In [ ]:
if "__interrupt__" in result:
    print("Paused. Approving email...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "reject"
                    }
                ]
            }
        ),
        config=config,
    )

    print(result["messages"][-1].content)

In [ ]:
result

{'messages': [HumanMessage(content='Send an email to john@example.com with the body: Hello, how are you?', additional_kwargs={}, response_metadata={}, id='73a444b1-2e49-4c4c-8ace-fa4031969786'),
  AIMessage(content='I’m ready to send the email, but I need a couple of details first:\n\n1. **Subject line** – what would you like the email’s subject to be?\n2. **Email ID** – the system requires an existing email ID (e.g., a draft ID) to send from. Could you provide that ID, or let me know if you’d like me to create a new draft for you?', additional_kwargs={'reasoning_content': 'The user wants to send an email to john@example.com with body "Hello, how are you?". We need to use send_email_tool. It requires email_id, cc, content, subject. We don\'t have subject. Probably we need to set a default subject? The user didn\'t specify subject. Could set subject empty or "No Subject". Also need email_id: maybe we need to create a new email? The tool expects email_id of an existing email? The descrip